## About
- This notebook includes training and evaluation of all two-feature combinations using symbolic regression

### Input
> All single feature and two-feature combination models
>
> performance metrics across training, test and validation sets of the above models

### Output
> Top 10 models based on AUROC in the test set
>
> Visualziations of model performance including hotspot proteins

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.metrics import roc_auc_score, roc_curve
from src.plots import plot_roc_curve
from datetime import datetime
import ast

In [ ]:
# Define project directory
Base = 'data_directory'

In [ ]:
# Load model performance
two_feature_combo_olink_ms=pd.read_excel(os.path.join(Base, 'symbolic_regression/models/model_performance_all_v3.xlsx'))

In [ ]:
df=two_feature_combo_olink_ms.copy()
train_p = df[df['set'].isin(['train', 'train_ms'])]
test_p = df[df['set'].isin(['test', 'test_ms'])]
test_sn_p = df[df['set'].isin(['test_sn'])]
validation_p = df[df['set'].isin(['validation1'])]
validation_sn_p = df[df['set'].isin(['validation1_sn'])]
validation_biop = df[df['set'].isin(['validation2'])]

In [ ]:
col_tokeep=['f1', 'feature1', 'feature2']

In [ ]:
titles = ['Performance in {} set'.format(i) for i in ['training', 'test', 'test_sn', 'validation1', 'validation_sn', 'validation2']] 
dfs = [train_p, test_p, test_sn_p, validation_p, validation_sn_p, validation_biop]

In [ ]:
score_metrics = ['roc_auc', 'f1','precision', 'recall', 'accuracy', 'balanced accuracy', 'specificity', 'mcc', 'NPV']

In [ ]:
import logging

logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger('fontTools').setLevel(logging.ERROR)

In [ ]:
date_str = datetime.now().strftime("%Y-%m-%d")
heatmap_datamatrix_file = os.path.join(Base, 'symbolic_regression/heatmaps/heatmap_datamatrix.xlsx')
with pd.ExcelWriter(heatmap_datamatrix_file) as writer:
    for score in score_metrics:
        col_tokeep = ['feature1', 'feature2'] + [score]
        for df, title in zip(dfs, titles):
            df_toplot = df[col_tokeep].pivot(index='feature1', columns='feature2', values=score).T
            #mask = np.triu(np.ones_like(df_toplot, dtype=bool))
            mask = df_toplot.isnull()
            plt.figure(figsize=(10,6))
            sns.heatmap(df_toplot, mask=mask, fmt='.2f', square=True, cmap='bwr', 
                        xticklabels=True, yticklabels=True, linewidth=1, linecolor='white', )
            plt.xlabel('')
            plt.ylabel('')
            plt.title('{}: {}'.format(title, score))
            df_toplot.to_excel(writer, sheet_name='{}_{}'.format(title.split(' ')[2], score))
            plt.rcParams['pdf.fonttype'] = 42
            plt.savefig(os.path.join(Base, 'symbolic_regression/heatmaps/{}_{}_{}.pdf'.format(title, score, date_str)))

#### Sensitivity analysis

In [ ]:
score='f1'
aa = validation_sn_p.set_index(['feature1', 'feature2'])[[score]]-validation_p.set_index(['feature1', 'feature2'])[[score]]

In [ ]:
aa.describe()

In [ ]:
subset='validation'
for score in score_metrics:
    title='{}: diff with super-healthy definition'.format(score.title())
    if subset=='validation':
        aa = validation_sn_p.set_index(['feature1', 'feature2'])[[score]]-validation_p.set_index(['feature1', 'feature2'])[[score]]
    elif subset=='test':
        aa = test_sn_p.set_index(['feature1', 'feature2'])[[score]]-test_p.set_index(['feature1', 'feature2'])[[score]]
    else:
        print('subset be either test or validation')
    aa = aa.reset_index().pivot(index='feature1', columns='feature2', values=score).T
    mask = aa.isnull()
    plt.figure(figsize=(10,6))
    sns.heatmap(aa, mask=mask, fmt='.2f', square=True, cmap='bwr', 
                xticklabels=True, yticklabels=True, linewidth=1, linecolor='white', center=0)
    plt.xlabel('')
    plt.ylabel('')
    plt.title(title)
    file_name=os.path.join(Base, 'symbolic_regression/heatmaps/sensitivity_analysis/{}_{}_diff_supernormal_{}.png'.format(score, subset, date_str))
    plt.rcParams['pdf.fonttype'] = 42
    plt.savefig('figures/temp_{}_{}.pdf'.format(subset, score))
    #plt.savefig(file_name, dpi=120, bbox_inches='tight')

#### supplementary table for super normal diff in model perforamnce

In [ ]:
sn_test_diff_scores = []
for score in score_metrics:  
    aa = test_sn_p.set_index(['feature1', 'feature2'])[[score]] - test_p.set_index(['feature1', 'feature2'])[[score]]
    sn_test_diff_scores.append(aa)
df_sn_test_diff_scores = pd.concat(sn_test_diff_scores, axis=1)

sn_validation_diff_scores = []
for score in score_metrics:  
    aa = validation_sn_p.set_index(['feature1', 'feature2'])[[score]] - validation_p.set_index(['feature1', 'feature2'])[[score]]
    sn_validation_diff_scores.append(aa)
df_sn_validation_diff_scores = pd.concat(sn_validation_diff_scores, axis=1)

In [ ]:
# Save results to ST8
with pd.ExcelWriter('supp_tables/ST8.xlsx') as writer:
    df_sn_test_diff_scores.reset_index().to_excel(writer, sheet_name='ST8-1', index=False)
    df_sn_validation_diff_scores.reset_index().to_excel(writer, sheet_name='ST8-2', index=False)

### Panel figure

In [ ]:
subset = 'test'

fig, axes = plt.subplots(3, 3, figsize=(24, 18))
plt.subplots_adjust(hspace=0.4, wspace=0.4)

for ax, score, label in zip(axes.flat, score_metrics, 'abcdefghi'):
    title = '{}: diff with super-healthy definition'.format(score.title())
    if subset == 'validation':
        aa = validation_sn_p.set_index(['feature1', 'feature2'])[[score]] - validation_p.set_index(['feature1', 'feature2'])[[score]]
    elif subset == 'test':
        aa = test_sn_p.set_index(['feature1', 'feature2'])[[score]] - test_p.set_index(['feature1', 'feature2'])[[score]]
    else:
        print('subset must be either test or validation')
        break

    aa = aa.reset_index().pivot(index='feature1', columns='feature2', values=score).T
    mask = aa.isnull()

    # Use gray colormap if all non-null values are zero
    all_zero = (aa[~mask].eq(0) | aa[~mask].isna()).all().all()
    cmap = 'gray' if all_zero else 'bwr'

    sns.heatmap(aa, mask=mask, fmt='.2f', square=True, cmap=cmap,
                xticklabels=True, yticklabels=True, linewidth=1, linecolor='white',
                center=0, ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, fontsize=10)
    ax.text(-0.1, 1.05, label + '.', transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top')

plt.rcParams['pdf.fonttype'] = 42
plt.savefig('figures/temp_{}_panel.pdf'.format(subset), bbox_inches='tight')

#### Prioritization of two-protein markers based on ROC-AUC in the test set

In [ ]:
matrix_train = pd.read_excel(heatmap_datamatrix_file, sheet_name='training_roc_auc').set_index('feature2')
matrix_test = pd.read_excel(heatmap_datamatrix_file, sheet_name='test_roc_auc').set_index('feature2')
matrix_validation = pd.read_excel(heatmap_datamatrix_file, sheet_name='validation1_roc_auc').set_index('feature2')

In [ ]:
#df_filtered = (matrix_train>0.8) & (matrix_test>0.8) & (matrix_validation>0.9)
df_filtered = (matrix_train>0.80) & (matrix_test>0.80) 
df_premium = df_filtered.replace({True:0.8, False:0.2})
df_toplot = df_premium
mask = np.triu(np.ones_like(df_premium, dtype=bool), k=1)
#df_toplot = df_premium.drop('Q08380').drop('Q08380', axis=1)

In [ ]:
df_filtered.sum().sum()

In [ ]:
from matplotlib.colors import ListedColormap
custom_cmap = ListedColormap(['lightgray', 'crimson'])

plt.figure(figsize=(10,6))
sns.heatmap(df_toplot, mask=mask, square=True, xticklabels=True, yticklabels=True, linewidth=1, 
            linecolor='white', cmap=custom_cmap, cbar=False,)
plt.rcParams['pdf.fonttype'] = 42
plt.ylabel('')
#plt.savefig(os.path.join(Base, 'symbolic_regression/heatmaps/prioritized.pdf'.format(title, score)), bbox_inches='tight')
#plt.savefig('temp.pdf', bbox_inches='tight', dpi=120)

In [ ]:
lala = matrix_validation[df_filtered]
lala[lala > 0.8].count().sum()

#### Top 20 models based on ROC-AUC in test set

In [ ]:
df_model_test = two_feature_combo_olink_ms[two_feature_combo_olink_ms['set'].isin(['test'])]

In [ ]:
def assign_rank(data_frame, column, ascending=False):
    df=data_frame.copy()
    df=df.sort_values(by=column, ascending=ascending)
    df['rank_{}'.format(column)]=np.arange(1, len(df)+1)
    return df

In [ ]:
df_model_test = assign_rank(df_model_test, 'roc_auc')
df_model_test['Model'] = 'Model_' + df_model_test['rank_roc_auc'].astype(str)

#### Radar plot

In [ ]:
# Extract top N models
n_models = 5
metric = 'roc_auc'
topn=df_model_test.sort_values(by=metric, ascending=False)[:n_models]
best5_features=df_model_test.sort_values(by=metric, ascending=False)[:n_models]['features'].tolist()
topn_features = best5_features + ["['alt_log2']", "['ast_log2']"]
df_performance=two_feature_combo_olink_ms.copy()

target_set = 'validation1'
df_toplot = df_performance[(df_performance['set']==target_set) & df_performance['features'].isin(topn_features)]

# Define the metrics
categories = ['roc_auc', 'f1', 'precision', 'NPV', 'recall', 'balanced accuracy', 'specificity', 'mcc',]
N = len(categories)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # Complete the loop

for i in best5_features:
    features_toinclude = [i] + ["['alt_log2']", "['ast_log2']"]
    df_toplot=df_performance[(df_performance['set']==target_set) & df_performance['features'].isin(features_toinclude)]
    # Initialize radar plot
    fig, ax = plt.subplots(figsize=(4, 3), subplot_kw=dict(polar=True))
    
    # Plot each model
    for idx, row in df_toplot.iterrows():
        values = row[categories].tolist()
        values += values[:1]  # Complete the loop
        ax.plot(angles, values, label=row['features'], linestyle='solid', marker='o', markersize=4,)
        #ax.fill(angles, values, alpha=0.1)
    
    # Format the plot
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    #ax.set_ylim(0.4, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'])
    ax.set_title("Comparative Performance of Top Models".format(n_models))
    ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1.1))
    plt.rcParams['pdf.fonttype'] = 42
    plt.tight_layout()
    plt.savefig('figures/temp_{}.pdf'.format(i), dpi=120, bbox_inches='tight')
    #plt.savefig(os.path.join(Base, 'symbolic_regression/figures/radar_plot_comparative_performance.pdf'), dpi=120, bbox_inches='tight')

### best performing two marker panel compared to AST 

In [ ]:
roc_curves_combined_file = os.path.join(Base, 'symbolic_regression/models/roc_curves_combined_v3.pkl')
with open(roc_curves_combined_file , 'rb') as f:
    roc_curves_combined = pickle.load(f)

In [ ]:
def plot_roc_curve(true_y, y_prob, color):
    """
    plots the roc curve based of the probabilities
    """

    fpr, tpr, thresholds = roc_curve(true_y, y_prob)
    plt.plot(fpr, tpr, color)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

In [ ]:
combo_selected = ('HGF','ast_log2')
#combo_selected = ('HGF','SCF')
splits_to_iter = ['train', 'test', 'test_sn', 'validation1']
roc_curves_selected = roc_curves_combined[combo_selected]
roc_curves_ast = roc_curves_combined['ast_log2']

for set_toplot in splits_to_iter:
    
    fig = plt.subplots(figsize=(4,4))
    plot_roc_curve(roc_curves_selected[set_toplot][0], roc_curves_selected[set_toplot][1], 
           color='steelblue')
    plot_roc_curve(roc_curves_ast[set_toplot.split('_')[0]][0], roc_curves_ast[set_toplot.split('_')[0]][1], color='gray')
    auroc_selected = roc_auc_score(roc_curves_selected[set_toplot][0], roc_curves_selected[set_toplot][1])
    auroc_ast = roc_auc_score(roc_curves_ast[set_toplot.split('_')[0]][0], roc_curves_ast[set_toplot.split('_')[0]][1])
    plt.plot([0, 1], [0, 1], ls="--", c='lightgray')
    plt.rcParams['pdf.fonttype'] = 42
    plt.title(set_toplot)
    plt.text(0.4, 0.2, 'ast auroc:{}'.format(round(auroc_ast, 2)), color='gray')
    plt.text(0.4, 0.1, '{}+{} auroc:{}'.format(combo_selected[0], combo_selected[1], round(auroc_selected, 2)), color='steelblue')
    # plt.savefig(os.path.join(Base, 'symbolic_regression/figures/best_{}_{}.pdf'.format(set_toplot, combo_selected)), 
    #        dpi=120, bbox_inches='tight')

### DeLong's test to compare two ROC-AUCs
> [DeLong](https://github.com/llniu/roc_comparison)

In [ ]:
from roc_comparison import compare_auc_delong_xu

ground_trouth = np.array([0,0,1,1,1])
pred_model_a  = np.array([0.1,0.2,0.6,0.7,0.8])
pred_model_b  = np.array([0.3,0.6,0.2,0.7,0.9])

log10_pvalue = compare_auc_delong_xu.delong_roc_test(ground_truth=ground_trouth,
                                      predictions_one=pred_model_a,
                                      predictions_two=pred_model_b
                                 )
assert np.round(10**log10_pvalue[0][0], 4) ==  0.3173 

In [ ]:
PAIR = combo_selected
TARGET_SET = 'test'
y_true = roc_curves_combined[PAIR][TARGET_SET][0]
y_pred_best = roc_curves_combined[PAIR][TARGET_SET][1]
if 'Q08380' in PAIR:
    y_pred_ast = roc_curves_combined['ast_log2_ms'][TARGET_SET][1]
    y_pred_alt = roc_curves_combined['alt_log2_ms'][TARGET_SET][1]
else:
    y_pred_ast = roc_curves_combined['ast_log2'][TARGET_SET][1]
    y_pred_alt = roc_curves_combined['alt_log2'][TARGET_SET][1]

In [ ]:
log10_pvalue = compare_auc_delong_xu.delong_roc_test(ground_truth=y_true,
                                      predictions_one=y_pred_best,
                                      predictions_two=y_pred_ast
                                 )

In [ ]:
log10_pvalue

In [ ]:
log10_pvalue = compare_auc_delong_xu.delong_roc_test(ground_truth=y_true,
                                      predictions_one=y_pred_best,
                                      predictions_two=y_pred_alt
                                 )

In [ ]:
log10_pvalue

#### Sensitivity analysis on supernormal only

In [ ]:
# 5 best performing models in the test set 
metric = 'roc_auc'
best5=df_model_test.sort_values(by=metric, ascending=False)[:5]
combo_to_iter = [tuple(ast.literal_eval(f)) for f in best5['features']]

In [ ]:
subsets = ['test', 'test_sn', 'validation1', 'validation1_sn']
metrics=['roc_auc', 'f1', 'precision', 'recall', 'NPV', 'balanced accuracy', 'specificity', 'mcc']


df_performance_top5 = df_performance[df_performance['features'].isin([str(list(i)) for i in combo_to_iter])]
df_performance_top5_sn = df_performance_top5[df_performance_top5['set'].isin(subsets)]
df_long = df_performance_top5_sn.melt(id_vars=['features', 'set'], value_vars=metrics, value_name='score', var_name='metric')

In [ ]:
fig, ax=plt.subplots(figsize=(6,4))
color_dict=dict(zip(subsets, ['steelblue', 'azure', 'crimson', 'lavenderblush']))
sns.barplot(x='metric', y='score', data=df_long, hue='set', palette=color_dict, edgecolor='gray')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.xlabel('')
plt.xticks(rotation=30, ha='right');
file_name=os.path.join(Base, 'symbolic_regression/heatmaps/sensitivity_analysis/diff_supernormal_top5_{}.png'.format(date_str))
plt.savefig(file_name, bbox_inches='tight', dpi=120)

In [ ]:
for combo in combo_to_iter:
    auroc_values={}
    fig, ax = plt.subplots(figsize=(4,4))
    for set_toplot in subsets:
        y_true=roc_curves_combined[combo][set_toplot][0]
        y_score=roc_curves_combined[combo][set_toplot][1]
        auroc=roc_auc_score(y_true, y_score)
        auroc_values[set_toplot]=auroc   # store in dict

        if 'sn' in set_toplot:
            plot_roc_curve(y_true, y_score, color='gray')
        else:
            plot_roc_curve(y_true, y_score, color='steelblue')

    plt.plot([0, 1], [0, 1], ls="--", c='lightgray')
    plt.rcParams['pdf.fonttype'] = 42
    plt.title(combo)

    # annotate AUROC values from the dictionary
    y0 = 0.1
    for i, (k, v) in enumerate(auroc_values.items()):
        ax.text(0.6, y0 + i*0.08, f"{k}: {v:.2f}",
                transform=ax.transAxes,
                fontsize=9, color='black',
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.6))


### Write to results

In [ ]:
sets_include = ['train', 'test', 'test_sn', 'validation1', 'validation1_sn']
df_st6 = df_performance[df_performance['set'].isin(sets_include)].drop(['Unnamed: 0'], axis=1)

In [ ]:
df_st6['top7']=np.where(df_st6['features'].isin(topn_features), True, False)

In [ ]:
set1 = df_st6[(df_st6['set']=='train') & (df_st6['roc_auc']>0.8)]['features']
set2 = df_st6[(df_st6['set']=='test') & (df_st6['roc_auc']>0.8)]['features']

In [ ]:
prioritized = set(set1) & set(set2)

In [ ]:
df_st6['prioritized']=np.where(df_st6['features'].isin(prioritized), True, False)

In [ ]:
with pd.ExcelWriter('supp_tables/ST6.xlsx') as writer:
    df_st6.to_excel(writer, sheet_name='ST6', index=False)

In [ ]:
df_st6[(df_st6['top7']) & (df_st6['set'].isin(['test', 'test_sn', 'validation1', 'validation1_sn']))]